# Lesson 4.3 — Hooks & Release Lifecycle

**Phase 4: Dependencies & Hooks**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Understand when each hook type fires in the release lifecycle
- Write `pre-install` and `post-upgrade` Job hooks
- Control hook execution order with `helm.sh/hook-weight`
- Configure hook cleanup with `helm.sh/hook-delete-policy`

---

## What Are Hooks?

Hooks let you run Kubernetes workloads (Jobs, Pods) at specific points in the release lifecycle — **before** or **after** install, upgrade, delete, or rollback.

Common uses:
- **pre-install:** seed a database, check prerequisites
- **post-install:** run smoke tests, send a notification
- **pre-upgrade:** run a database migration
- **pre-delete:** drain connections, take a backup

> **Reference:** [Chart Hooks](https://helm.sh/docs/topics/charts_hooks/) — https://helm.sh/docs/topics/charts_hooks/

---

## The Hook Lifecycle for `helm install`

```
helm install foo
    |
    ├─ CRDs installed
    ├─ Templates rendered
    ├─ pre-install hooks fired (and waited on)
    ├─ Regular resources applied to Kubernetes
    ├─ post-install hooks fired (and waited on)
    └─ Release object returned
```

---

## Available Hooks

| Hook | Fires When |
|---|---|
| `pre-install` | After template rendering, before resources are created |
| `post-install` | After all resources are created |
| `pre-upgrade` | After template rendering, before resources are updated |
| `post-upgrade` | After all resources are updated |
| `pre-rollback` | After template rendering, before rollback |
| `post-rollback` | After rollback |
| `pre-delete` | Before any resources are deleted |
| `post-delete` | After all resources are deleted |
| `test` | When `helm test` is run |

---

## Writing a Hook

A hook is a normal template file with a special annotation. **Convention:** place hooks in `templates/hooks/`.

In [ ]:
rm -rf /tmp/hookdemo && mkdir -p /tmp/hookdemo/templates/hooks

cat > /tmp/hookdemo/Chart.yaml << 'EOF'
apiVersion: v2
name: hookdemo
version: 0.1.0
appVersion: "1.0"
EOF

cat > /tmp/hookdemo/values.yaml << 'EOF'
image:
  repository: alpine
  tag: "3.19"
migration:
  enabled: true
EOF

echo "Setup complete"

In [ ]:
%%writefile /tmp/hookdemo/templates/hooks/pre-install-migrate.yaml
{{- if .Values.migration.enabled }}
apiVersion: batch/v1
kind: Job
metadata:
  name: {{ .Release.Name }}-db-migrate
  labels:
    app: {{ .Chart.Name }}
  annotations:
    # This annotation is what makes this a hook — without it, this is just a Job
    "helm.sh/hook": pre-install,pre-upgrade

    # Hook weight controls execution order among multiple hooks
    # Lower weight = runs first. Range: any integer.
    "helm.sh/hook-weight": "-5"

    # Delete the Job before creating a new one (avoids immutable field conflicts)
    "helm.sh/hook-delete-policy": before-hook-creation
spec:
  # ttlSecondsAfterFinished: 120    # Optional: auto-cleanup after 2 minutes
  template:
    metadata:
      name: {{ .Release.Name }}-db-migrate
    spec:
      restartPolicy: Never
      containers:
        - name: migrate
          image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
          command:
            - /bin/sh
            - -c
            - |
              echo "Running database migration for release {{ .Release.Name }}..."
              echo "App version: {{ .Chart.AppVersion }}"
              echo "Migration complete!"
{{- end }}

In [ ]:
# Render the hook template — note the hook annotations
helm template my-release /tmp/hookdemo \
  --show-only templates/hooks/pre-install-migrate.yaml

---

## Hook Delete Policies

| Policy | When is the hook resource deleted? |
|---|---|
| `before-hook-creation` | Delete the old hook before creating a new one (default) |
| `hook-succeeded` | Delete the hook resource after it completes successfully |
| `hook-failed` | Delete the hook resource if it fails |

You can combine policies: `"helm.sh/hook-delete-policy": before-hook-creation,hook-succeeded`

> **Important:** Hook resources are **not** part of the release — `helm uninstall` will NOT delete them unless you set `hook-delete-policy`. Always set a delete policy to avoid resource accumulation.

In [ ]:
%%writefile /tmp/hookdemo/templates/hooks/post-install-notify.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: {{ .Release.Name }}-post-install-notify
  annotations:
    "helm.sh/hook": post-install
    "helm.sh/hook-weight": "0"
    # Delete after success — keeps the cluster tidy
    "helm.sh/hook-delete-policy": hook-succeeded
spec:
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: notify
          image: alpine:3.19
          command:
            - /bin/sh
            - -c
            - |
              echo "=== Post-install: {{ .Release.Name }} deployed successfully ==="
              echo "Namespace: {{ .Release.Namespace }}"
              echo "Chart: {{ .Chart.Name }}-{{ .Chart.Version }}"
              # In a real scenario: call a webhook, send a Slack message, etc.

In [ ]:
# Render all templates and see hooks alongside regular resources
helm template my-release /tmp/hookdemo | grep -E 'kind:|name:|helm.sh/hook'

---

## Hook Weights and Ordering

When multiple hooks fire at the same lifecycle event, they are sorted:
1. By weight (ascending) — negative numbers run first
2. By resource kind
3. By name (alphabetically)

In [ ]:
# Demonstrate ordering with a schema check running before migration
cat > /tmp/hookdemo/templates/hooks/pre-install-schema-check.yaml << 'EOF'
apiVersion: batch/v1
kind: Job
metadata:
  name: {{ .Release.Name }}-schema-check
  annotations:
    "helm.sh/hook": pre-install,pre-upgrade
    # Weight -10 runs BEFORE the migration (weight -5)
    "helm.sh/hook-weight": "-10"
    "helm.sh/hook-delete-policy": before-hook-creation
spec:
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: check
          image: alpine:3.19
          command: ["/bin/sh", "-c", "echo 'Schema check passed'"]
EOF

echo "Hooks ordered by weight:"
echo ""
echo "  Weight -10: schema-check (runs first)"
echo "  Weight  -5: db-migrate   (runs second)"
echo "  Weight   0: post-install-notify (post-install event)"

---

## Testing a Hook (Dry Run vs Live)

`helm template` shows hooks just like regular resources. To test a real hook:

```bash
# Install and watch the Job complete before the main resources are applied
helm install my-release /tmp/hookdemo \
  --namespace hook-demo \
  --create-namespace

# Watch the hook Job
kubectl get jobs -n hook-demo -w
kubectl logs job/my-release-db-migrate -n hook-demo
```

In [ ]:
# Verify the full rendered chart — hooks appear with their annotations
helm template my-release /tmp/hookdemo

---

## Exercise

In [ ]:
# TASK 1: Add a 'pre-delete' hook that logs a message
#         'Taking backup before deleting release {{ .Release.Name }}'.
#         Set hook-delete-policy to hook-succeeded.
# TODO:


# TASK 2: Add a conditional: only run the migration hook if
#         .Values.migration.enabled is true (it currently always runs).
#         Test both cases with helm template.
# TODO:


# TASK 3: Write a post-upgrade hook (weight 5) that prints
#         the new appVersion after an upgrade.
#         It should fire on post-upgrade AND post-install.
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- Hooks are Kubernetes resources with `"helm.sh/hook": <event>` annotation.
- Most hooks are Jobs that run to completion — Helm waits for them before proceeding.
- Use `helm.sh/hook-weight` to control execution order (lower = earlier).
- Always set `helm.sh/hook-delete-policy` — otherwise hook resources accumulate in the cluster.
- Common hook pairs: `pre-install` + `post-install`, `pre-upgrade` + `post-upgrade`.

---

## Next Lesson

➡️ [Lesson 5.1 — OCI Registries & Chart Publishing](../phase-5-real-world/lesson-5.1-oci-registries.ipynb)